In [49]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated

In [51]:
class State(TypedDict):
    topic: str      
    research: str    
    draft: str      
    final: str

In [52]:
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [53]:
def ask(system: str, user: str) -> str:
    """Single LLM call helper."""
    r = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return r.content

In [54]:
def researcher(state: State) -> dict:
    out = ask(
        "You are a researcher. List 4-6 concise key facts/points. Bullets only.",
        f"Topic: {state['topic']}",
    )
    return {"research": out}

In [55]:
def writer(state: State) -> dict:
    out = ask(
        "You are a writer. Turn the research into a clear, engaging short article.",
        f"Topic: {state['topic']}\n\nResearch:\n{state['research']}",
    )
    return {"draft": out}

In [56]:
def editor(state: State) -> dict:
    out = ask(
        "You are an editor. Tighten, fix grammar/flow, return only the polished final copy.",
        f"Draft:\n{state['draft']}",
    )
    return {"final": out}

In [57]:
g = StateGraph(State)
g.add_node("researcher", researcher)
g.add_node("writer", writer)
g.add_node("editor", editor)

g.add_edge(START, "researcher")
g.add_edge("researcher", "writer")
g.add_edge("writer", "editor")
g.add_edge("editor", END)

graph = g.compile()

In [58]:
result = graph.invoke({"topic": "business strategies for small business owners in Africa"})

print("=== RESEARCH ===\n", result["research"], "\n")
print("=== DRAFT ===\n", result["draft"], "\n")
print("=== FINAL ===\n", result["final"])

=== RESEARCH ===
 - Leverage digital technology to expand reach and optimize operations, utilizing e-commerce platforms and social media marketing.
- Adopt a customer-centric approach by understanding local needs and preferences to tailor products and services accordingly.
- Build strong networks and partnerships with local suppliers, other businesses, and community organizations to enhance support and resources.
- Focus on sustainability and social responsibility to build brand loyalty, attract investment, and meet growing consumer demand for ethical practices.
- Utilize financial tools and resources, such as microfinancing and mobile payment systems, to improve cash flow management and access capital. 

=== DRAFT ===
 **Empowering Small Businesses in Africa: Strategic Approaches for Success**

In an ever-evolving global market, small business owners in Africa face unique challenges and immense potentials. By adopting strategic approaches tailored to their specific contexts, entrepren

In [59]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

In [60]:
memory = InMemorySaver()

In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

In [62]:
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]


In [63]:
from langchain_core.messages import SystemMessage
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [72]:
def chat_handler(state: RouterState) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

In [73]:
builder = StateGraph(RouterState)

In [74]:
       
builder.add_node("chat", chat_handler)

In [75]:
builder.add_edge(START, "chat")

In [76]:

builder.add_edge("chat", END)

In [77]:
router = builder.compile()

In [78]:
out = router.invoke({"messages": [HumanMessage(content="Pelumi is a beautician")]})

In [79]:
print(out["messages"][-1].content)

That sounds wonderful! As a beautician, Pelumi likely has skills in areas such as skincare, haircare, makeup application, and possibly even nail care. Is there something specific you would like to know or discuss about Pelumi’s work?
